# RAD-DINO baseline on CheXpert — Colab

Trains RAD-DINO (`microsoft/rad-dino`, a DINOv2-based chest X-ray encoder from Hugging Face) on CheXpert. Unlike the other baseline notebooks, RAD-DINO isn't a `timm` model -- it's loaded via the `transformers` library and is encoder-only (no classification head), so patch 11 in cell 7 adds a small wrapper class (`RadDinoClassifier`) with a plain linear head on top, and patch 12 adds RAD-DINO's own normalization stats to the dataloader.

**Before running:**
1. `Runtime > Change runtime type` → GPU. This notebook runs RAD-DINO at **224×224**, matching every other baseline notebook, for a controlled, consistent comparison -- not RAD-DINO's native 518×518. RAD-DINO handles this fine technically (its position embeddings auto-interpolate to the smaller 16x16 patch grid, verified locally, no errors), but it's worth knowing this moves the model off the resolution it was actually pretrained at, which can somewhat reduce the advantage its pretraining would otherwise provide. If you want RAD-DINO evaluated at its native resolution instead, set `args.input_size = args.img_size = 518` in cell 12 (and lower `BATCH_SIZE` back down -- see the comment there).
2. RAD-DINO is always initialized from its own Hugging Face pretrained weights (`AutoModel.from_pretrained("microsoft/rad-dino")`, auto-downloaded) -- there's no meaningful "random init" variant, since the whole point of using this baseline is its pretraining.

Cell 12 has a `NUM_CLASSES` toggle: **14** (current default) trains the head on all 14 CheXpert labels but only reports on the 5 competition classes; **5** trains directly on just those 5.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# Colab's pre-installed stack (pandas, etc.) is compiled against numpy>=2's actual C ABI --
# this isn't just a pip-metadata preference, running them against numpy<2 crashes for real
# ("numpy.dtype size changed, may indicate binary incompatibility"). We previously tried
# downgrading numpy to dodge an *unrelated* numpy-2.0 incompatibility in imgaug (a transitive
# dep of albumentations) -- that broke pandas instead, and the downgrade didn't even take
# effect this-session anyway, since Colab pre-imports numpy before any of our cells run. So:
# make sure numpy is restored to >=2 here (undoing any earlier accidental downgrade left on
# disk), and fix the imgaug incompatibility directly instead (see section 2, patches 4 & 5).
import sys
!{sys.executable} -m pip install -q --break-system-packages "numpy>=2"
import numpy
print('numpy', numpy.__version__, '@', numpy.__file__)
print('kernel python:', sys.executable)

## 1. Kaggle credentials + dataset download

In [ ]:
import os, json, getpass, sys

kaggle_username = input('Kaggle username: ')
kaggle_key = getpass.getpass('Kaggle API key: ')

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
del kaggle_key

!{sys.executable} -m pip install -q --break-system-packages kaggle
os.makedirs('/content/data', exist_ok=True)

# Using the kaggle package's own Python API instead of any CLI form -- it turns out this
# version of the package has no `__main__.py` (`-m kaggle` fails with "cannot be directly
# executed"), and the bare `kaggle` console script hits the same PATH-mismatch class of
# issue we saw with pip earlier. Calling the API in-process sidesteps all of that.
import kaggle
kaggle.api.authenticate()
# quiet=False -- dataset_download_files defaults to quiet=True, which is why this ran with
# zero output before. With it off, kaggle's own tqdm byte-progress bar (MB downloaded, speed,
# ETA) shows up instead of the download appearing to hang silently for several minutes.
kaggle.api.dataset_download_files('ashery/chexpert', path='/content/data', unzip=True, quiet=False)
print('Download complete.')

In [ ]:
# The train.csv/valid.csv paths are prefixed with 'CheXpert-v1.0-small/', but the
# Kaggle extract puts train/ and valid/ directly under /content/data. Symlink to bridge it.
import os
data_root = '/content/data'
link = os.path.join(data_root, 'CheXpert-v1.0-small')
if not os.path.islink(link) and not os.path.exists(link):
    os.symlink(data_root, link)

print(sorted(os.listdir(data_root)))
import pandas as pd
df = pd.read_csv(os.path.join(data_root, 'train.csv'))
print(df.shape, list(df.columns))

## 2. Clone the CheXpert finetuning codebase, patch it, and install dependencies
Reuses an existing open-source CheXpert finetuning pipeline (dataloader, training loop, metrics) -- not any pretrained model from it, just its well-tested training code. Same 2021-era dependency pins as the other baseline notebooks in this project (`timm==0.5.4`, `albumentations==0.4.3`; `transformers` is also already in the install list, needed for RAD-DINO), same patches:

- `np.bool` alias fix, dead import removal (patches 1-2)
- `class_indices` support for training on just the 5 competition classes (patch 3)
- Real bf16 `torch.autocast` mixed precision (patch 6)
- `meanRecall`/`meanAccuracy` metrics, threaded through both reporting paths (patches 7-8)
- Clean per-epoch results CSV (patch 9) and per-epoch weights-only checkpoints (patch 10)

Plus two new patches specific to this notebook:
- **Patch 11**: adds a `RadDinoClassifier` wrapper (RAD-DINO's encoder + a linear head) and wires it in as `model_name="rad_dino"`, since the model-selector has no support for non-`timm`/Hugging-Face models by default.
- **Patch 12**: adds RAD-DINO's own normalization mean/std (from its `preprocessor_config.json`) as a `normalize="rad_dino"` option in the dataloader.

This cell is safe to re-run (e.g. after a kernel restart) -- the clone and patches are skipped/no-op if already present.

In [ ]:
import os, sys

# Install the same dependencies the training code needs (this doesn't change: copying our own
# code below doesn't install third-party packages).
!{sys.executable} -m pip install -q --break-system-packages timm==0.5.4 albumentations==0.4.3 opencv-python-headless scikit-image scikit-learn \
    simpleitk scipy pydicom yacs einops transformers pretrainedmodels imgaug pillow tqdm

# imgaug (a transitive dep of albumentations==0.4.3, old and unmaintained) has a np.sctypes
# reference (removed in numpy 2.0) in parameters.py, plus a dead `from scipy import misc` import
# (removed in modern scipy) in the same file. Both crash on *import*, so we can't `import imgaug`
# first to locate the file -- that import is exactly what crashes. Using importlib.util.find_spec
# instead, which locates the module without executing it.
import importlib.util, os as _os
_ia_dir = _os.path.dirname(importlib.util.find_spec('imgaug').origin)
_ia_params = _os.path.join(_ia_dir, 'parameters.py')
_s = open(_ia_params).read()
_before = _s
_s = _s.replace('np.sctypes["float"]', '[np.float16, np.float32, np.float64]')
_s = _s.replace('from scipy import misc\n', '')
if _s != _before:
    open(_ia_params, 'w').write(_s)
    print('imgaug patched:', _ia_params)
else:
    print('imgaug already patched (or patterns not found), skipping:', _ia_params)
import imgaug
print('imgaug import OK, version', imgaug.__version__)

# The training code (dataloader/trainer/engine/models) comes from GitHub at a fixed tag, so every baseline
# runs byte-identical code whichever Google account runs the notebook (the older copy lived on one account's
# Drive and had to be re-synced by hand). The md5 check stops here if any file differs by a single byte from
# the code that was verified (an uninterrupted run is byte-identical to the engine Ark+ trained with).
import subprocess, shutil, hashlib
REPO_URL = "https://github.com/fei7fei/chexpert-baselines.git"
CODE_TAG = "protocol1-baselines-v1"
EXPECTED_CODE_MD5 = {
    "engine.py": "c5b6e179169c35481be1d6e54c9179af",
    "trainer.py": "3b368affdd29b1b03475f82474ebfa6a",
    "models.py": "e0ced0f809680aaebfc97ec3fd6d0e14",
    "dataloader.py": "6aba3a804bb10625a49c9027734a7b76",
    "utils.py": "a7c17deb55d24e826c74b46ad848c441",
}
if not os.path.exists('/content/chexpert-baselines'):
    subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", CODE_TAG, REPO_URL, "/content/chexpert-baselines"], check=True)
if not os.path.exists('/content/chexpert_code'):
    shutil.copytree('/content/chexpert-baselines/Repos/ark_plus_repo/Ark_Plus/Finetuning', '/content/chexpert_code')
for _fn, _md5 in EXPECTED_CODE_MD5.items():
    _got = hashlib.md5(open(os.path.join('/content/chexpert_code', _fn), 'rb').read()).hexdigest()
    assert _got == _md5, f"{_fn}: md5 {_got} != {_md5} -- not the verified protocol code, stop"
sys.path.insert(0, '/content/chexpert-baselines/analysis')  # paper_outputs.py: paper tables and figures
%cd /content/chexpert_code
print(f'Training code {CODE_TAG} loaded from GitHub; all {len(EXPECTED_CODE_MD5)} files match the verified md5s.')

In [ ]:
import torch
print('torch', torch.__version__, 'cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0))
from transformers import AutoModel
m = AutoModel.from_pretrained("microsoft/rad-dino")
print('RAD-DINO encoder built OK, params:', sum(p.numel() for p in m.parameters()))
del m

## 3. Mount Google Drive for checkpoint persistence
`/content` is ephemeral — if the runtime disconnects or gets recycled (e.g. Colab Compute Units running out), everything in it is gone, checkpoints included. This mounts Drive and points training's save location there instead, so progress survives even a full runtime loss. Re-running cell 12 after a restart will then automatically resume from the last saved epoch (via `args.resume`) instead of starting over.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_CKPT_DIR = '/content/drive/MyDrive/raddino_chexpert_checkpoints'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print('Checkpoints and results will be saved to:', DRIVE_CKPT_DIR)

## 4. Run finetuning on the CheXpert benchmark
Trains RAD-DINO per the `NUM_CLASSES` toggle set in the cell below, reporting AUROC/MCC/AP/F1/Recall/Accuracy on the 5 competition classes each epoch either way. Adjust `BATCH_SIZE`/`EPOCHS`/`PATIENCE` there for your GPU and time budget -- defaults mirror the settings used for the other baseline notebooks' diagnostic runs (generous epoch ceiling, patience effectively disabled, proportionally-scaled warmup/cooldown, 224px input, same batch size).

In [ ]:
%cd /content/chexpert_code
import sys, os
sys.path.insert(0, '.')

# Checkpoints and results must land on Google Drive, not the runtime's own disk (lost when it disconnects).
assert os.path.ismount('/content/drive'), "Google Drive is not mounted -- run the Drive cell (section 3) and authorise it first"

_old_argv = sys.argv
sys.argv = ['main_classification.py']
from main_classification import get_args_parser
args = get_args_parser()
sys.argv = _old_argv

# ===== Frozen protocol v1 -- identical across all 6 models, fix this before running anything =====
INPUT_SIZE = 320                    # native CheXpert-small short side -- avoids gratuitous up/downsampling
UNCERTAIN_LABEL = "Zeros"           # U-Zeros for main results; flip to "Ones" for the appendix sensitivity check
EPOCHS = 30
SEEDS_SCREENING = [13]              # 1 seed: cheap ranking pass on the 20% ablation subset
SEEDS_HEADLINE = [13, 42]     # 3 seeds: confirmation pass on the full training pool

args.model_name = "rad_dino"
args.init = "rad_dino"  # label only -- RadDinoClassifier always loads microsoft/rad-dino; args.init is not read for this model
args.pretrained_weights = None
args.input_size = INPUT_SIZE
args.img_size = INPUT_SIZE
args.num_class = 14   # train the head on all 14 CheXpert labels; report only on the 5 competition
                       # classes via test_diseases below (original Ark+ protocol convention)
args.data_set = "CheXpert"
args.data_dir = "/content/data"
args.key = "teacher"
# args.batch_size: set in the "Batch" block below (64 images per optimizer step, as Ark+; RAD-DINO used 32 until 2026-11-05)
args.epochs = EPOCHS
args.patience = EPOCHS           # no early stopping -- every model trains the full 50 epochs so run
                                  # lengths stay comparable across the whole comparison
args.opt = "adamw"
args.lr = 5e-5
args.weight_decay = 0.05
args.sched = "cosine"
args.warmup_epochs = 1
args.cooldown_epochs = 0
args.uncertain_label = UNCERTAIN_LABEL
args.test_every_epoch = True
args.save_every_epoch_checkpoint = True
args.amp_dtype = "fp16"
args.device = "cuda"
args.resume = True
args.workers = 8  # bumped from 2: a rock-steady per-batch time with zero jitter is the signature of a CPU-bound dataloader, not a slow GPU

# ===== Batch: 64 images per optimizer step, as Ark+ (never change the effective batch) =====
# If the GPU can't hold 64 images at once, each step is split into micro-batches with gradient accumulation:
# 2 x 32 gives the same gradient as 1 x 64 (verified against the original trainer; exact for LayerNorm
# models). The values used are recorded in the env_*.json file written below.
import torch
EFFECTIVE_BATCH = 64
MICRO_BATCH = 64  # if this runs out of GPU memory, set 32 (the effective batch stays 64)
args.batch_size = MICRO_BATCH
args.grad_accum_steps = EFFECTIVE_BATCH // MICRO_BATCH
assert MICRO_BATCH * args.grad_accum_steps == EFFECTIVE_BATCH, "MICRO_BATCH must divide 64"
print(f"GPU {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB): "
      f"{args.grad_accum_steps} x {MICRO_BATCH} = {EFFECTIVE_BATCH} images per optimizer step")

diseases_14 = ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
               'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax',
               'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']
competition_5 = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion']
test_diseases_5 = [diseases_14.index(c) for c in competition_5]

from dataloader import CheXpert, build_transform_classification, build_protocol_splits, filter_csv_by_patients
from engine import classification_engine

# Patient-level held-out test (carved from train) + stratified 20% ablation subset -- computed
# once and cached to a *shared* Drive folder (not this model's own checkpoint folder) so every
# one of the 6 models' notebooks reads the exact same patient partition, in whatever order they
# happen to run. That's what makes the cross-model comparison valid rather than 6 separate
# anecdotes on 6 different splits.
PROTOCOL_SHARED_DIR = '/content/drive/MyDrive/chexpert_protocol_v1'
os.makedirs(PROTOCOL_SHARED_DIR, exist_ok=True)

train_csv = os.path.join(args.data_dir, "train.csv")
splits = build_protocol_splits(train_csv, competition_5, cache_path=os.path.join(PROTOCOL_SHARED_DIR, "protocol_split.csv"))

def _cached_filter(name, patient_set):
    p = os.path.join(PROTOCOL_SHARED_DIR, f"{name}.csv")
    if not os.path.exists(p):
        filter_csv_by_patients(train_csv, patient_set, p)
    return p

train_pool_csv = _cached_filter("train_pool", splits["train_pool"])
held_out_test_csv = _cached_filter("held_out_test", splits["held_out_test"])
ablation_subset_csv = _cached_filter("ablation_subset", splits["ablation_subset"])
official_val_csv = os.path.join(args.data_dir, "valid.csv")

# ===== Split guard: must be exactly the dataset and patient split Ark+ was trained and tested on =====
# The fingerprints were computed on 2026-11-06 from CheXpert-small train.csv with the protocol's own split
# code, which reproduces Ark+'s held-out gt.csv row for row. Each covers every image path and all 14 raw
# labels, in order. A mismatch means a different dataset, Kaggle version or split, and a model trained on it
# would not be comparable to Ark+ -- so stop here instead of training.
import hashlib
import pandas as pd
def split_fingerprint(csv_path):
    _d = pd.read_csv(csv_path)
    _lab = _d[diseases_14].fillna(9).astype(int).astype(str)
    return len(_d), hashlib.md5("\n".join(_d["Path"] + "," + _lab.agg(",".join, axis=1)).encode()).hexdigest()
ARKPLUS_SPLIT = {  # name: (images, fingerprint)
    "train_pool": (190359, "fd210ea10a3f5c364de5d167b57144f3"),
    "held_out_test": (33055, "03d149b5a3504245e7e9a2f42a1c2e9f"),
    "ablation_subset": (38186, "87c42c824823e16386ee0cbc026202db"),
    "official_valid": (234, "bca0b7e132dfc73e7aab492cf40cc084"),
}
_split_paths = {"train_pool": train_pool_csv, "held_out_test": held_out_test_csv,
                "ablation_subset": ablation_subset_csv, "official_valid": official_val_csv}
split_fingerprints = {}
for _name, _p in _split_paths.items():
    split_fingerprints[_name] = split_fingerprint(_p)
    assert split_fingerprints[_name] == ARKPLUS_SPLIT[_name], \
        f"{_name}: {split_fingerprints[_name]} != Ark+'s {ARKPLUS_SPLIT[_name]} -- not Ark+'s dataset/split, do not train"
print("Split check passed: same images, labels and patient split as Ark+.")

args.train_list = train_pool_csv
args.val_list = official_val_csv     # the official 234-study set -- used for per-epoch threshold
                                      # tuning/checkpoint selection only, never for final reporting
args.test_list = held_out_test_csv   # final reporting -- held out from train, never seen in training

def make_ds(mode, file_path):
    return CheXpert(images_path=args.data_dir, file_path=file_path,
        augment=build_transform_classification(normalize=args.normalization, mode=mode, crop_size=args.input_size, resize=args.img_size),
        uncertain_label=args.uncertain_label, unknown_label=args.unknown_label)

dataset_val = make_ds("valid", args.val_list)
dataset_test = make_ds("test", args.test_list)
print(f"val (official, threshold tuning): {len(dataset_val)}  |  test (held-out, final report): {len(dataset_test)}")

model_path = os.path.join(DRIVE_CKPT_DIR, "Models/Classification", args.data_set)
output_path = os.path.join(DRIVE_CKPT_DIR, "Outputs/Classification", args.data_set)

# ===== Record this session's software versions and a fingerprint of the training code =====
# All 6 models must run byte-identical training code; these md5s let anyone check that afterwards.
import json, hashlib, datetime, timm, transformers, sklearn, numpy, torch
_now = datetime.datetime.now(datetime.timezone.utc)
_env = {
    "utc": _now.isoformat(timespec="seconds"),
    "model_name": args.model_name, "amp_dtype": args.amp_dtype, "code_tag": CODE_TAG,
    "micro_batch": args.batch_size, "grad_accum_steps": args.grad_accum_steps,
    "effective_batch": args.batch_size * args.grad_accum_steps,
    "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
    "split_fingerprints": split_fingerprints,
    "torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0),
    "timm": timm.__version__, "transformers": transformers.__version__,
    "numpy": numpy.__version__, "sklearn": sklearn.__version__,
    "code_md5": {_fn: hashlib.md5(open(_fn, "rb").read()).hexdigest()
                 for _fn in ["engine.py", "trainer.py", "models.py", "dataloader.py", "utils.py"]},
}
os.makedirs(output_path, exist_ok=True)
with open(os.path.join(output_path, f"env_{args.model_name}_{_now:%Y%m%dT%H%M%SZ}.json"), "w") as _fh:
    json.dump(_env, _fh, indent=1)
print(json.dumps(_env, indent=1))

# ===== Tier 1: screening -- cheap, single seed, ranks variants on the 20% subset =====
print("\n=== TIER 1: screening (1 seed, 20% stratified subset) ===")
args.exp_name = "_protocol1_raddino_screening"
args.num_trial = len(SEEDS_SCREENING)
args.start_index = 0
args.seed_list = SEEDS_SCREENING
dataset_train_screen = make_ds("train", ablation_subset_csv)
print(f"screening train size: {len(dataset_train_screen)}")
classification_engine(args, model_path, output_path, diseases_14, dataset_train_screen, dataset_val, dataset_test, test_diseases=test_diseases_5)

# ===== Tier 2: headline -- 3 seeds, full training data. Report both tiers; the two-tier split =====
# ===== is itself part of what makes ~20 runs comparable rather than 20 anecdotes. =====
print("\n=== TIER 2: headline (3 seeds, full training pool) ===")
args.exp_name = "_protocol1_raddino_headline"
args.num_trial = len(SEEDS_HEADLINE)
args.start_index = 0
args.seed_list = SEEDS_HEADLINE
dataset_train_full = make_ds("train", train_pool_csv)
print(f"headline train size: {len(dataset_train_full)}")
classification_engine(args, model_path, output_path, diseases_14, dataset_train_full, dataset_val, dataset_test, test_diseases=test_diseases_5)

## 5. Results
Per-epoch AUROC prints during training above. Final summary (mean/std AUC, MCC, AP, F1 across the 5 classes) lands in `Outputs/Classification/CheXpert/*_results.txt`; checkpoints in `Models/Classification/CheXpert/` — both now inside `DRIVE_CKPT_DIR` on your Google Drive (see cell 10), so they survive a runtime loss. If cell 12 gets cut off mid-run, just re-run it — `args.resume=True` picks back up from the last saved epoch automatically.

The next cell shows the clean per-epoch table (Epoch, train_loss, val_loss, F1, AUC, recall, accuracy) as a DataFrame — patch 9 (cell 7) writes this to its own CSV alongside the verbose text log, rewritten after every epoch so it's readable even from a run that got interrupted partway through.

In [ ]:
!find "{DRIVE_CKPT_DIR}/Outputs" -type f
!echo ---
!cat "{DRIVE_CKPT_DIR}/Outputs/Classification/CheXpert"/*_results.txt

In [ ]:
import glob, os

def _resolve_latest_epoch_table():
    try:
        _exact = glob.glob(f"{DRIVE_CKPT_DIR}/Models/Classification/CheXpert/{args.exp_name}/{args.exp_name}_run_*_epoch_table.csv")
        if _exact:
            return sorted(_exact)[-1]
    except NameError:
        pass
    _csvs = glob.glob(f"{DRIVE_CKPT_DIR}/Models/Classification/CheXpert/*/*_epoch_table.csv")
    if not _csvs:
        return None
    return max(_csvs, key=os.path.getmtime)
import pandas as pd

_latest = _resolve_latest_epoch_table()
if _latest is None:
    print("No epoch table found yet -- run cell 12 through at least one epoch first.")
else:
    print("Showing:", _latest)
    display(pd.read_csv(_latest))

## 5b. Loss curves (over/underfitting check) and per-epoch checkpoints
The cell below pulls out just epochs 20, 35, and 50 into their own table. The plot after that reads the same `epoch_table.csv` and plots `train_loss` vs `val_loss` across epochs -- train loss dropping while val loss flattens or rises is the standard overfitting signal; both staying high/flat together points to underfitting.

The second cell lists the per-epoch checkpoint files written by patch 10 (only present if `SAVE_EVERY_EPOCH_CHECKPOINT=True` in cell 12) and shows how to load a specific epoch's weights back into a model for inspection or inference, independent of whichever epoch ended up "best" for resuming training.

In [ ]:
import glob, os

def _resolve_latest_epoch_table():
    try:
        _exact = glob.glob(f"{DRIVE_CKPT_DIR}/Models/Classification/CheXpert/{args.exp_name}/{args.exp_name}_run_*_epoch_table.csv")
        if _exact:
            return sorted(_exact)[-1]
    except NameError:
        pass
    _csvs = glob.glob(f"{DRIVE_CKPT_DIR}/Models/Classification/CheXpert/*/*_epoch_table.csv")
    if not _csvs:
        return None
    return max(_csvs, key=os.path.getmtime)
import pandas as pd

_latest = _resolve_latest_epoch_table()
if _latest is None:
    print("No epoch table found yet -- run cell 12 through at least one epoch first.")
else:
    print("Showing:", _latest)
    _df = pd.read_csv(_latest)

    _epochs_of_interest = [20, 35, 50]
    _subset = _df[_df["Epoch"].isin(_epochs_of_interest)]
    _missing = sorted(set(_epochs_of_interest) - set(_subset["Epoch"]))
    if _missing:
        print(f"Not reached yet: epoch(s) {_missing} (latest epoch so far: {_df['Epoch'].max()})")
    if not _subset.empty:
        display(_subset.set_index("Epoch"))

In [ ]:
import glob, os

def _resolve_latest_epoch_table():
    try:
        _exact = glob.glob(f"{DRIVE_CKPT_DIR}/Models/Classification/CheXpert/{args.exp_name}/{args.exp_name}_run_*_epoch_table.csv")
        if _exact:
            return sorted(_exact)[-1]
    except NameError:
        pass
    _csvs = glob.glob(f"{DRIVE_CKPT_DIR}/Models/Classification/CheXpert/*/*_epoch_table.csv")
    if not _csvs:
        return None
    return max(_csvs, key=os.path.getmtime)
import pandas as pd
import matplotlib.pyplot as plt

_latest = _resolve_latest_epoch_table()
if _latest is None:
    print("No epoch table found yet -- run cell 12 through at least one epoch first.")
else:
    print("Showing:", _latest)
    _df = pd.read_csv(_latest)

    _best_val_loss_epoch = int(_df.loc[_df["val_loss"].idxmin(), "Epoch"])
    _best_val_loss = _df["val_loss"].min()
    _best_auc_epoch = int(_df.loc[_df["AUC"].idxmax(), "Epoch"])
    _best_auc = _df["AUC"].max()
    print(f"Lowest val_loss:  epoch {_best_val_loss_epoch} (val_loss={_best_val_loss:.4f})")
    print(f"Highest val AUC:  epoch {_best_auc_epoch} (AUC={_best_auc:.4f})")

    # Plot 1: train vs val loss, with the lowest-val_loss epoch marked -- the classic
    # overfitting signal is train_loss still falling while val_loss flattens or climbs
    # past this point.
    fig, ax1 = plt.subplots(figsize=(7, 4))
    ax1.plot(_df["Epoch"], _df["train_loss"], marker="o", label="train_loss")
    ax1.plot(_df["Epoch"], _df["val_loss"], marker="o", label="val_loss")
    ax1.axvline(_best_val_loss_epoch, color="gray", linestyle="--", linewidth=1)
    ax1.annotate(f"min val_loss\nepoch {_best_val_loss_epoch}",
                 xy=(_best_val_loss_epoch, _best_val_loss), xytext=(10, 10),
                 textcoords="offset points", fontsize=9, color="gray")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.set_title("Train vs Val Loss (over/underfitting check)")
    plt.tight_layout()
    plt.show()

    # Plot 2: val AUC as the primary line (threshold-independent, the actual task-quality
    # metric) with its max marked; F1/Recall/Accuracy shown lighter as secondary context,
    # since they depend on a per-epoch re-picked threshold and are noisier signals.
    fig2, ax2 = plt.subplots(figsize=(7, 4))
    ax2.plot(_df["Epoch"], _df["AUC"], marker="o", linewidth=2.5, color="C0", label="AUC")
    for _col, _c in zip(["F1", "recall", "accuracy"], ["C1", "C2", "C3"]):
        ax2.plot(_df["Epoch"], _df[_col], marker=".", linewidth=1, alpha=0.5, color=_c, label=_col)
    ax2.axvline(_best_auc_epoch, color="gray", linestyle="--", linewidth=1)
    ax2.annotate(f"max AUC\nepoch {_best_auc_epoch}",
                 xy=(_best_auc_epoch, _best_auc), xytext=(10, -15),
                 textcoords="offset points", fontsize=9, color="gray")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Score")
    ax2.legend(loc="lower right")
    ax2.set_title("Val AUC (primary) vs F1 / Sen(Recall) / Accuracy per epoch")
    plt.tight_layout()
    plt.show()

In [ ]:
import glob

_ckpts = sorted(glob.glob(f"{DRIVE_CKPT_DIR}/Models/Classification/CheXpert/*/*_epoch*.pth.tar"))
print("Per-epoch checkpoints found:")
for f in _ckpts:
    print(" ", f)

# Example: load one specific epoch's weights (edit the index/path below).
# import torch
# from models import build_classification_model
# ckpt = torch.load(_ckpts[0], map_location="cpu")
# print("epoch:", ckpt["epoch"], "val_loss:", ckpt["val_loss"])
# model = build_classification_model(args)
# model.load_state_dict(ckpt["state_dict"])

## 6. Paper tables and figures (saved to Drive)
Saved next to this model's results on Drive, in `Outputs/Classification/CheXpert/paper/`, for each tier:
- `*_metrics_per_seed.csv`: AUC, F1, sensitivity, accuracy, MCC, AP and ECE per seed, plus training-length numbers (best-AUC epoch, lowest-validation-loss epoch, final losses);
- `*_metrics_per_class.csv`: the same metrics for each of the 5 competition classes;
- `*_summary_range.csv`: the min–max range over seeds in the submitted Table 3's format (never mean ± std);
- `*_curves.pdf` / `.png`: train vs validation loss and validation AUC per epoch, with the checkpointed (best-AUC) epoch marked.

F1 / sensitivity / accuracy / MCC here are `engine.py`'s (Youden threshold fit on the test labels), the same kind as Table 3. The validation-tuned Figure 1 numbers come from `AllModels_Gaps_Colab.ipynb`. Afterwards, copy this model's Drive folder into `results/<model>/` on the Mac; `code/analysis/paper_outputs.py` builds the cross-model figures (all six models, Ark+ vs Swin-L) from there.

In [ ]:
import glob
from IPython.display import Image, display
from paper_outputs import save_experiment_outputs

MODEL_KEY = "raddino"
PAPER_DIR = os.path.join(output_path, "paper")
for _tier in ["screening", "headline"]:
    save_experiment_outputs(MODEL_KEY, _tier, model_path, output_path, PAPER_DIR)
for _png in sorted(glob.glob(os.path.join(PAPER_DIR, "*_curves.png"))):
    display(Image(_png))